# RunPeaknet

Goal:
- Load my trained Peaknet model
- Run it on spectra
- Plot what it predicts


## Step 1 - Imports



In [ ]:
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/peaknet_matplotlib')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## Step 2 - Paths

I set the paths to the data folder, model folder, and checkpoint.

It is advised to use a dataset which is not the dataset it traind on.


In [ ]:
# Get the current working directory
base_dir = os.getcwd()

# Get the parent dir
project_dir = os.path.dirname(base_dir)

# Defining the data and models directories
data_dir = os.path.join(project_dir, 'Data', 'dataset_small')
models_dir = os.path.join(project_dir, 'Models')

# Defining the checkpoint path
checkpoint_dir = os.path.join(models_dir, 'checkpoints')
checkpoint_name = 'peaknet_20260611_003618.pth'
checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)

print('Base dir:', base_dir)
print('Data dir:', data_dir)
print('Models dir:', models_dir)
print('Checkpoint:', checkpoint_path)
print('Checkpoint exists:', os.path.exists(checkpoint_path))


## Step 3 - Load the test data

I load a set number of spectra, set as max_examples.

I use peak spectra here because they have masks, so I can compare the model prediction against the target.


In [ ]:
# Number of spectra to load
max_examples = 50

# Load the spectra and masks
spectra_all = np.load(os.path.join(data_dir, 'peaks_spectra.npy'), allow_pickle=True)
masks_all = np.load(os.path.join(data_dir, 'peaks_masks.npy'), allow_pickle=True)

# Cut down to the sample size I selected
spectra = np.array(spectra_all[:max_examples], dtype=object)
masks = np.array(masks_all[:max_examples], dtype=object)

# Clip each spectrum separately because the spectra can have different lengths
def clip_spectra_array(array, low, high):
    
    clipped = []
    for spectrum in array:
        clipped.append(np.clip(spectrum, low, high).astype(np.float32))
    return np.array(clipped, dtype=object)

# Clip and convert to float32 for PyTorch
spectra = clip_spectra_array(spectra, -0.2, 1.0)
masks = clip_spectra_array(masks, 0, 1.0)


print('Spectra shape:', spectra.shape)
print('Masks shape  :', masks.shape)
print('First spectrum length:', len(spectra[0]))
print('Spectra min/max:', min(s.min() for s in spectra), max(s.max() for s in spectra))
print('Masks min/max  :', min(m.min() for m in masks), max(m.max() for m in masks))


## Step 4 - Import the same model architecture

The checkpoint only stores the trained weights.

So I must create the same model shape first, then put the saved weights into it.


In [ ]:
# Add the models directory to the Python path so I can import the model code
sys.path.insert(0, models_dir)

# Import the peaknet model used to train the last model checkpoint
from TrainPeaknet_2026_06_01 import Peaknet

# Load the model
print('Loaded model code from TrainPeaknet_2026_06_01.py')
model = Peaknet()
# Output the model architecture
print('Model architecture:')
model


## Step 5 - Load the trained weights

This takes the saved weights from the checkpoint and puts them into the model.

Then I use `model.eval()` because I am testing the model, not training it.


In [ ]:
# Set device to GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Send model to device
model = Peaknet().to(device)

# Load the model checkpoint
state_dict = torch.load(checkpoint_path, map_location=device)
# Load the state dict into the model
model.load_state_dict(state_dict)
model.eval()

print('Using device:', device)
print('Loaded checkpoint:', checkpoint_path)


## Step 6 - Add the channel dimension

The model expects input shape:

`(number of spectra, channels, spectrum length)`

So I add the channel dimension just like in the training notebook.


In [ ]:
# Adding a batch dimension and channel dimension to the input tensor
x = torch.tensor(spectra[0][None, None, :]).to(device)

print('Example input tensor shape:', x.shape)


## Step 7 - Run the model

I use `torch.no_grad()` because I only want predictions.

I do not need gradients because I am not updating the model weights.


In [ ]:
predictions = []

# The conv kernels are 51 wide, so each output sample sees 25 samples of context
# on either side. At the array edges that context is faked by the conv's internal
# zero padding, which the model reads as a sharp edge and predicts as a peak.
# To avoid that, I pad the input myself with 25 zeros on each side, run inference,
# then crop those 25 samples off so the prediction lines up with the original
# spectrum (and with the masks used for the MSE).
pad = 25

# Run one spectrum at a time because the spectra can have different lengths
with torch.no_grad():
    for spectrum in spectra:
        padded = np.pad(spectrum, pad, mode='constant', constant_values=0).astype(np.float32)
        x = torch.tensor(padded[None, None, :]).to(device)
        prediction = model(x)
        # Crop the padded border back off so the length matches the input spectrum
        predictions.append(prediction[0, 0, pad:-pad].cpu().numpy())

predictions = np.array(predictions, dtype=object)

print('Number of predictions:', len(predictions))
print('First prediction shape:', predictions[0].shape)


## Step 8 - Calculate a simple error

Mean squared error is used to get one simple number for how close the prediction is to the target mask.


In [ ]:
mse_values = []

# Compute the MSE for each spectrum separately because lengths can differ
for prediction, mask in zip(predictions, masks):
    mse_values.append(np.mean((prediction - mask) ** 2))

mse = np.mean(mse_values)

print('MSE:', mse)


## Step 9 - Plot one prediction

Plot:
- the input spectrum
- the target mask
- the model prediction


In [ ]:
example_index = 0

plt.figure(figsize=(10, 5))
plt.xlim(0, len(spectra[example_index]))
plt.plot(spectra[example_index], label='input spectrum')
plt.plot(masks[example_index], label='target mask')
plt.plot(predictions[example_index], label='prediction')
plt.xlabel('Bin')
plt.ylabel('Value')
plt.title('Peaknet prediction')
plt.legend()
plt.show()


## Step 10 - Plot and Save all predictions

Plot:
- the input spectrum
- the target mask
- the model prediction


In [ ]:
# Get the inference dir
inference_dir = os.path.join(base_dir, 'Inference')
os.makedirs(inference_dir, exist_ok=True)

# Save one plot for each prediction
for index in range(1, min(max_examples, len(spectra))):

    plt.figure(figsize=(10, 5))
    plt.plot(spectra[index], label='input spectrum')
    plt.plot(masks[index], label='target mask')
    plt.plot(predictions[index], label='prediction')
    plt.xlabel('Bin')
    plt.ylabel('Value')
    plt.title('Peaknet prediction')
    plt.legend()
    plt.savefig(os.path.join(inference_dir, f'peaknet_prediction_{index:03d}.png'))
    plt.close()
